# GPU Execution Notebook — Mechanistic Interpretability

This notebook runs the full pipeline on Colab GPU:
1. Setup & install
2. Generate datasets
3. Capture activations (8 layers: 4,8,12,16,20,24,28,32)
4. Train SAEs
5. Build attribution graphs
6. Run intervention experiments (full-knockout diagnostic, scan, inhibit, swap)

**Runtime settings**: GPU (any tier), High-RAM recommended.

In [1]:
# Step 1: Mount Google Drive and extract project
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Set to True if you uploaded the project.zip directly to Colab files sidebar:
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar.")

Mounted at /content/drive
Extracting /content/drive/MyDrive/mphil-project/project.zip...
Current working directory: /content
total 148
drwxr-xr-x 2 root root  4096 Jul  8 00:50 configs
drwxr-xr-x 3 root root  4096 Jul  8 00:50 data
drwx------ 5 root root  4096 Jul  8 00:50 drive
-rw-r--r-- 1 root root   264 Jul  7 02:58 environment.yml
-rw-r--r-- 1 root root 19186 Jul  7 02:58 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root  1579 Jul  7 02:58 Instructions.md
drwxr-xr-x 3 root root  4096 Jul  8 00:50 mechanistic_data
drwxr-xr-x 2 root root  4096 Jul  8 00:50 mechanistic_interpretability_repro.egg-info
drwxr-xr-x 3 root root  4096 Jul  8 00:50 outputs
-rw-r--r-- 1 root root  6551 Jul  7 02:58 project.txt
drwxr-xr-x 2 root root  4096 Jul  8 00:50 __pycache__
-rw-r--r-- 1 root root 11435 Jul  7 23:15 README.md
-rw-r--r-- 1 root root 13981 Jul  7 23:15 run_gpu.ipynb
-rw-r--r-- 1 root root 43650 Jul  7 02:58 run.ipynb
drwxr-xr-x 1 root root  4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root

In [2]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

Obtaining file:///content
  Preparing metadata (setup.py) ... done
  Attempting uninstall: mechanistic-interpretability-repro
    Found existing installation: mechanistic-interpretability-repro 0.1.0
    Uninstalling mechanistic-interpretability-repro-0.1.0:
      Successfully uninstalled mechanistic-interpretability-repro-0.1.0
  Running setup.py develop for mechanistic-interpretability-repro


In [3]:
# Step 3: Generate Datasets
!python data/generate_datasets.py --capitals
print("\nDataset files:")
!ls -lh data/*.csv

Generating Datasets...

Flag '--capitals' captured. Building structural sentences for geography data...
Saved 1000 capital entries with text templates.
Standalone Mode complete:
Saved 1000 addition problems and 
Saved 1000 unit problems.

Dataset files:
-rw-r--r-- 1 root root  52K Jul  8 00:51 data/addition_data.csv
-rw-r--r-- 1 root root 6.3K Jul  7 02:58 data/capitals_base.csv
-rw-r--r-- 1 root root 6.3K Jul  7 02:58 data/capitals_base_old.csv
-rw-r--r-- 1 root root  97K Jul  8 00:51 data/capitals_data.csv
-rw-r--r-- 1 root root 151K Jul  8 00:51 data/units_data.csv
-rw-r--r-- 1 root root  30K Jul  7 02:58 data/units_data_old.csv


In [4]:
# Step 4: Capture Activations for 8 layers
# This takes ~5-10 minutes on a T4/V100/A100
!python src/capture_activations.py \
  --output-dir mechanistic_data \
  --layers 4 8 12 16 20 24 28 32 \
  --seed 787

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 194.49it/s]
Loading prompts for behaviour: capitals
Loading prompts for behaviour: addition
Loading prompts for behaviour: units
Total prompts loaded: 3000
Running forward passes to capture activations...
100% 3000/3000 [12:40<00:00,  3.95it/s]
Saving activation tensors to disk...
Saved activations for layer 4: (3000, 2560) to /content/mechanistic_data/activations_layer4.npy
Saved activations for layer 8: (3000, 2560) to /content/mechanistic_data/activations_layer8.npy
Saved activations for layer 12: (3000, 2560) to /content/mechanistic_data/activations_layer12.npy
Saved activations for layer 16: (3000, 2560) to /content/mechanistic_data/activations_layer16.npy
Saved activations for layer 20: (3000, 2560) to /content/mechanistic_data/activations_layer20.npy
Saved activations for layer 24: (3000,

In [5]:
# Step 5: Train SAEs on all 8 layers
# This takes ~10-20 minutes on GPU depending on hardware
!python src/train.py --config configs/sae_config.yaml
print("\nSAE checkpoint files:")
!ls -lh mechanistic_data/sae_checkpoints/

Using CUDA device: Tesla T4

=== Training SAE for layer 4 ===
Layer 4 activation scaling factor (raw mean L2 / sqrt(d_in)): 0.1519
Epoch 00 | train_loss=0.3863 | val_loss=0.0500 | mse=0.0497 | l1=0.3273
Epoch 01 | train_loss=0.0295 | val_loss=0.0145 | mse=0.0142 | l1=0.2347
Epoch 02 | train_loss=0.0107 | val_loss=0.0090 | mse=0.0087 | l1=0.2208
Epoch 03 | train_loss=0.0080 | val_loss=0.0079 | mse=0.0077 | l1=0.2155
Epoch 04 | train_loss=0.0105 | val_loss=0.0100 | mse=0.0098 | l1=0.2038
Epoch 05 | train_loss=0.0091 | val_loss=0.0091 | mse=0.0089 | l1=0.1888
Epoch 06 | train_loss=0.0067 | val_loss=0.0065 | mse=0.0063 | l1=0.1788
Epoch 07 | train_loss=0.0051 | val_loss=0.0050 | mse=0.0049 | l1=0.1732
Epoch 08 | train_loss=0.0044 | val_loss=0.0047 | mse=0.0045 | l1=0.1685
Epoch 09 | train_loss=0.0041 | val_loss=0.0044 | mse=0.0042 | l1=0.1629
Epoch 10 | train_loss=0.0038 | val_loss=0.0042 | mse=0.0040 | l1=0.1588
Epoch 11 | train_loss=0.0038 | val_loss=0.0040 | mse=0.0039 | l1=0.1538
Epoch

In [9]:
!ls mechanistic_data/
!cp -r mechanistic_data/* /content/drive/MyDrive/mphil-project/

activation_metadata.csv  activations_layer4.npy
activations_layer12.npy  activations_layer8.npy
activations_layer16.npy  sae_checkpoints
activations_layer20.npy  train_indices.npy
activations_layer24.npy  train_val_indices_per_layer.npy
activations_layer28.npy  val_indices.npy
activations_layer32.npy


---
## Phase 5: Attribution Graphs

Build attribution graphs for different prompts. The graph identifies which SAE features
are most important for predicting the next token.

In [10]:
# Graph 1: Capital of state (Texas/Dallas -> Austin)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target "Austin" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/dallas_graph.json \
  --output-html outputs/dallas_graph.html \
  --output-mermaid outputs/dallas_graph.md

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 190.72it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/mechanistic_data/sae_checkpoints to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1519)
Loaded SAE for layer 8 (scaling factor: 0.3033)
Loaded SAE for layer 12 (scaling factor: 0.2863)
Loaded SAE for layer 16 (scaling factor: 0.3016)
Loaded SAE for layer 20 (scaling factor: 0.3414)
Loaded SAE for layer 24 (scaling factor: 0.6633)
Loaded SAE for layer 28 (scaling factor: 1.0271)
Loaded SAE for layer 32 (scaling factor: 2.0917)
Running forward pass...

Top 5 model predictions:
  ' after' (prob: 0.2471, logit: 16.62)
  ' Austin' (prob: 0.1924, logit: 16.38)
  ' Dallas' (prob: 0.1924, logit: 16.38)
  ' what' (prob: 0.1030, logit: 15.75)
  ' for' (prob: 0.0552, logit:

In [11]:
# Graph 2: Capital of country (Afghanistan/Kandahar -> Kabul)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target "Kabul" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/kabul_graph.json \
  --output-html outputs/kabul_graph.html \
  --output-mermaid outputs/kabul_graph.md

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.61it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/mechanistic_data/sae_checkpoints to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1519)
Loaded SAE for layer 8 (scaling factor: 0.3033)
Loaded SAE for layer 12 (scaling factor: 0.2863)
Loaded SAE for layer 16 (scaling factor: 0.3016)
Loaded SAE for layer 20 (scaling factor: 0.3414)
Loaded SAE for layer 24 (scaling factor: 0.6633)
Loaded SAE for layer 28 (scaling factor: 1.0271)
Loaded SAE for layer 32 (scaling factor: 2.0917)
Running forward pass...

Top 5 model predictions:
  ' Kabul' (prob: 0.9844, logit: 21.50)
  ' K' (prob: 0.0085, logit: 16.75)
  ' as' (prob: 0.0017, logit: 15.12)
  ' after' (prob: 0.0012, logit: 14.81)
  ' in' (prob: 0.0004, logit: 13.56)



In [12]:
# Graph 3: A lower-confidence example — better for showing intervention effects
# Delaware/Wilmington -> Dover (model may be less confident here)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Wilmington is named" \
  --target "Dover" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/dover_graph.json \
  --output-html outputs/dover_graph.html \
  --output-mermaid outputs/dover_graph.md

Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.28it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/mechanistic_data/sae_checkpoints to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1519)
Loaded SAE for layer 8 (scaling factor: 0.3033)
Loaded SAE for layer 12 (scaling factor: 0.2863)
Loaded SAE for layer 16 (scaling factor: 0.3016)
Loaded SAE for layer 20 (scaling factor: 0.3414)
Loaded SAE for layer 24 (scaling factor: 0.6633)
Loaded SAE for layer 28 (scaling factor: 1.0271)
Loaded SAE for layer 32 (scaling factor: 2.0917)
Running forward pass...

Top 5 model predictions:
  ' Raleigh' (prob: 0.2949, logit: 16.00)
  ' what' (prob: 0.2158, logit: 15.69)
  ' Charlotte' (prob: 0.1152, logit: 15.06)
  ' Wilmington' (prob: 0.0898, logit: 14.81)
  ' after' (prob: 0.0

---
## Phase 6: Intervention Experiments

### Diagnostic 1: Full MLP Knockout
First, verify the hooked layers actually matter by zeroing out their MLP outputs entirely.
If this doesn't change the prediction, the layers we chose are uninvolved.

In [14]:
# Full knockout: proves the layers matter (upper bound on intervention effect)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --full-knockout \
  --output outputs/knockout_dallas.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.58it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[3/3] Running Full MLP Knockout (zeroing MLP output at last token for layers [4, 8, 12, 16, 20, 24, 28, 32])...
  - Top prediction: ' after' (prob: 0.5742,

In [15]:
# Full knockout on Kabul prompt
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --full-knockout \
  --output outputs/knockout_kabul.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 193.71it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[3/3] Running Full MLP Knockout (zeroing MLP output at last token for layers [4, 8, 12, 16, 20, 24, 28, 32])...
  - Top prediction: ' Kabul' (prob: 0.9492, logit: 22.12)
  - Target ' as

### Diagnostic 2: Progressive Ablation Scan

Ablate features progressively (top-10, 25, 50, 100, ALL from the graph)
to see how many features need to be removed before the logit shifts.

In [16]:
# Scan: progressive ablation using features from the attribution graph
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --graph-json outputs/dallas_graph.json \
  --scan \
  --output outputs/scan_dallas.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 194.72it/s]
Loaded 172 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[3/3] Running Progressive Ablation Scan...
  Total features from 

In [17]:
# Scan on Kabul prompt
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --graph-json outputs/kabul_graph.json \
  --scan \
  --output outputs/scan_kabul.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 192.76it/s]
Loaded 175 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[3/3] Running Progressive Ablation Scan...
  Total features from graph: 160
  Scan levels: [10,

In [18]:
# Scan on Dover prompt (likely lower confidence -> easier to observe effects)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Wilmington is named" \
  --target-token "Dover" \
  --graph-json outputs/dover_graph.json \
  --scan \
  --output outputs/scan_dover.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 198.37it/s]
Loaded 172 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Raleigh' (prob: 0.4277, logit: 16.38)
  - Target ' Charlotte': prob: 0.1152, logit: 15.06
  - Target 'D': prob: 0.0000, logit: -0.75
  - Target ' Dover': prob: 0.0014, logit: 10.62
  - Target ' Raleigh': prob: 0.4277, logit: 16.38
  - Target ' what': prob: 0.1475, logit: 15.31

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Raleigh' (prob: 0.4277, logit: 16.38)
  - Target ' Charlotte': prob: 0.1152, logit: 15.06
  - Target 'D': prob: 0.0000, logit: -0.75
  - Target ' Dover': prob: 0.0014, logit: 10.62
  - Target ' Raleigh': prob: 0.4277, logit: 16.38
  - Target ' what

### Experiment 1: Targeted Inhibition

Use the features identified by the attribution graph. The `--graph-json` flag
automatically extracts all active features from the pruned graph.

In [19]:
# Inhibition with ALL features from the attribution graph
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --graph-json outputs/dallas_graph.json \
  --output outputs/inhibit_dallas_full.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 193.45it/s]
Loaded 172 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2051, logit: 16.12

[3/3] Running Inhibition Intervention (inhibited features: {4: [5

In [20]:
# Inhibition on Kabul
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --graph-json outputs/kabul_graph.json \
  --output outputs/inhibit_kabul_full.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 197.42it/s]
Loaded 175 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition: {}...
  - Top prediction: ' Kabul' (prob: 0.9883, logit: 21.75)
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target 'K': prob: 0.0000, logit: 7.88
  - Target ' K': prob: 0.0059, logit: 16.62
  - Target ' Kabul': prob: 0.9883, logit: 21.75

[3/3] Running Inhibition Intervention (inhibited features: {4: [5165, 2960, 6878, 2587, 2136, 3

### Experiment 2: Activation Swap-In

Swap features from a source prompt (where model predicts X) into a target prompt
(where model predicts Y). If the swapped features encode X's identity, the target
prompt should shift toward predicting X.

In [21]:
# Swap: Take features from Oakland (Sacramento) and swap into Dallas (Austin)
# Hypothesis: if location-identity features are swapped, Dallas prompt shifts toward Sacramento
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --graph-json outputs/dallas_graph.json \
  --target-token "Sacramento, Austin" \
  --output outputs/swap_oakland_to_dallas.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 191.74it/s]
Loaded 172 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target ' Dallas': prob: 0.2051, logit: 16.12
  - Target 'Sac': prob: 0.0000, logit: -6.19
  - Target ' Sacramento': prob: 0.0000, logit: 2.75
  - Target ' San': prob: 0.0000, logit: 7.72

Source baseline prediction on 'Fact: The capital of the state containing Oakland is named':
  - Target ' after': prob: 0.4629, logit: 18.62
  - Target 'Austin': prob: 0.0000, logit: -0.91
  - Target ' Austin': prob: 0.0000, logit: 7.22
  - Target ' Dallas': prob: 0.0000, logit: 5.81
  - Target

In [22]:
# Swap: Full latent swap (no --features, swaps everything) — strongest signal
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Sacramento, Austin" \
  --output outputs/swap_full_oakland_to_dallas.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 193.15it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2637, logit: 16.38)
  - Target ' after': prob: 0.1245, logit: 15.62
  - Target ' Austin': prob: 0.2637, logit: 16.38
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2051, logit: 16.12
  - Target 'Sac': prob: 0.0000, logit: -6.19
  - Target ' Sacramento': prob: 0.0000, logit: 2.75
  - Target ' San': prob: 0.0000, logit: 7.72

Source baseline prediction on 'Fact: The capital of the state containing Oakland is named':
  - Target ' after': prob: 0.4629, logit: 18.62
  - Target ' Austin': prob: 0.0000, logit: 7.22
  - Target 'Austin': prob: 0.0000, logit: -0.91
  - Target ' Dallas': prob: 0.0000, logit: 5.81
  - Target 'Sac': prob: 0.0000, logit: 7.06
  - Target ' Sacramento': prob: 0.1699, logit: 17.62
  -

In [23]:
# Swap across countries: Kandahar (Kabul) -> Berlin prompt
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Kandahar is named" \
  --prompt "Fact: The capital of the country containing Munich is named" \
  --graph-json outputs/kabul_graph.json \
  --target-token "Kabul, Berlin" \
  --output outputs/swap_kandahar_to_munich.json

Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 194.39it/s]
Loaded 175 nodes from graph JSON; extracted features across 8 layers (160 total features)

[1/3] Clean Model Baseline:
  - Top prediction: ' Munich' (prob: 0.7070, logit: 19.12)
  - Target ' after': prob: 0.1084, logit: 17.25
  - Target ' as': prob: 0.0057, logit: 14.31
  - Target 'Berlin': prob: 0.0000, logit: 6.19
  - Target ' Berlin': prob: 0.0242, logit: 15.75
  - Target 'K': prob: 0.0000, logit: -3.41
  - Target ' K': prob: 0.0000, logit: 5.28
  - Target ' Kabul': prob: 0.0000, logit: 0.84
  - Target ' Munich': prob: 0.7070, logit: 19.12
  - Target ' what': prob: 0.0845, logit: 17.00

Source baseline prediction on 'Fact: The capital of the country containing Kandahar is named':
  - Target ' after': prob: 0.0009, logit: 14.75
  - Target ' as': prob: 0.0012, logit: 15.00
  - Target 'Berlin': pr

---
## Save Outputs to Drive

In [25]:
# Copy all outputs to Google Drive for persistence
import shutil
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in os.listdir('/content/outputs'):
    src = f'/content/outputs/{f}'
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print(f'Copied {f}')

# Also copy SAE checkpoints
drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
os.makedirs(drive_sae, exist_ok=True)
sae_dir = '/content/mechanistic_data/sae_checkpoints'
if os.path.isdir(sae_dir):
    for f in os.listdir(sae_dir):
        shutil.copy2(f'{sae_dir}/{f}', drive_sae)
    print(f'\nCopied SAE checkpoints to {drive_sae}')
print('Done!')

Copied kabul_graph.json
Copied kabul_graph.md
Copied dover_graph.json
Copied inhibit_dallas_full.json
Copied swap_oakland_to_dallas.json
Copied scan_dallas.json
Copied scan_dover.json
Copied swap_full_oakland_to_dallas.json
Copied dallas_graph.json
Copied dallas_graph.html
Copied kabul_graph.html
Copied knockout_kabul.json
Copied dover_graph.html
Copied inhibit_kabul_full.json
Copied dallas_graph.md
Copied swap_kandahar_to_munich.json
Copied dover_graph.md
Copied scan_kabul.json
Copied knockout_dallas.json

Copied SAE checkpoints to /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints
Done!


---
## Results Summary

After running, report back the following information:

1. **Full knockout results**: did zeroing out all 8 MLP layers change the top prediction?
   - Copy the printed output from the `--full-knockout` cells.

2. **Scan results**: at what level of progressive ablation does the logit shift become noticeable?
   - Copy the logit deltas printed by the `--scan` cells.

3. **Swap results**: did the full latent swap shift the prediction toward the source capital?
   - Copy the target token probabilities from the swap cells.

4. **Any errors or unexpected behavior** during execution.

This information tells us:
- If knockout has no effect → the layers are wrong or the model uses other pathways
- If scan shows gradual decline → features are working correctly but distributed
- If full swap shifts output → SAE features encode identity information correctly

In [28]:
!ls -l mechanistic_data/sae_checkpoints/

total 2079248
-rw-r--r-- 1 root root  98304128 Jul  8 01:07 latents_layer12.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:08 latents_layer16.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:08 latents_layer20.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:09 latents_layer24.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:10 latents_layer28.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:10 latents_layer32.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:06 latents_layer4.npy
-rw-r--r-- 1 root root  98304128 Jul  8 01:06 latents_layer8.npy
-rw-r--r-- 1 root root      9657 Jul  8 01:07 sae_layer12_metadata.json
-rw-r--r-- 1 root root 167817989 Jul  8 01:07 sae_layer12.pt
-rw-r--r-- 1 root root      9647 Jul  8 01:08 sae_layer16_metadata.json
-rw-r--r-- 1 root root 167817989 Jul  8 01:08 sae_layer16.pt
-rw-r--r-- 1 root root      9656 Jul  8 01:08 sae_layer20_metadata.json
-rw-r--r-- 1 root root 167817989 Jul  8 01:08 sae_layer20.pt
-rw-r--r-- 1 root root      9653 Jul  8 01:09 sae_layer24_met